모델배포기초  
Last modified : 2026.03    
제작 : 박광석 (모두의연구소)  

# Day 8 — 자율 프로젝트: 나만의 모델 서빙 서비스 만들기

---

> **오늘의 목표**
>
> Day 1~7에서 배운 기술을 조합하여, 본인이 관심 있는 도메인의 모델을 서빙하는 서비스를 직접 만듭니다.  
> 따라하기가 아닌, **스스로 설계하고 구현하는** 첫 경험입니다.

---



## 1. 프로젝트 요구사항

---

### 1.1 필수 조건

Day 5(주택 가격 예측)와 Day 6~7(이미지 분류 / 챗봇)에서 만든 서비스와 **동일한 구조**를 가져야 합니다.

```
필수 구현 항목:

1. FastAPI 백엔드
   - 추론 엔드포인트 (POST /predict)
   - Pydantic으로 입력 검증
   - 비동기 추론 (run_in_executor)

2. API Key 인증
   - Day 6의 auth.py 재사용

3. Streamlit 프론트엔드
   - 사용자 입력 → API 호출 → 결과 표시

4. 에러 처리
   - 잘못된 입력, 모델 에러 시 적절한 HTTP 상태 코드 반환
```



### 1.2 제한 사항

```
하지 않는 것:

- 모델 학습 (사전학습 모델을 가져다 씁니다)
- Docker 패키징 (MLOps 과정에서 다룹니다)
- 데이터베이스 연동
```



### 1.3 평가 기준

```
✅ 서버가 정상적으로 실행되는가?
✅ Swagger UI에서 추론이 동작하는가?
✅ API Key 없이 요청하면 401이 반환되는가?
✅ 잘못된 입력에 대해 적절한 에러 메시지가 나오는가?
✅ Streamlit UI에서 입력 → 결과 확인이 가능한가?
```

---

## 2. 모델 선택 가이드

---

### 2.1 Hugging Face에서 모델 찾기

[Hugging Face Models](https://huggingface.co/models)에서 사전학습 모델을 선택합니다.
모델 학습은 하지 않고, `from_pretrained()`으로 바로 사용할 수 있는 모델을 고릅니다.

**모델 선택 시 확인할 것:**

```
1. 태스크가 명확한가? (text-classification, image-classification, summarization 등)
2. 한국어를 지원하는가? (필수는 아니지만, 데모가 직관적입니다)
3. 모델 크기가 적당한가? (CPU 환경이면 500MB 이하를 권장합니다)
4. pipeline()으로 바로 사용 가능한가?
```



### 2.2 도메인별 추천 예시

아래는 예시일 뿐입니다. **본인이 관심 있는 도메인을 자유롭게 선택하세요.**

```
┌────────────────┬───────────────────────────┬──────────────────────────┐
│ 도메인          │ 태스크                     │ 추천 모델 (예시)          │
├────────────────┼───────────────────────────┼──────────────────────────┤
│ 감정 분석       │ text-classification       │ snunlp/KR-FinBert-SC    │
│ 뉴스 분류       │ text-classification       │ 원하는 분류 모델          │
│ 텍스트 요약     │ summarization             │ eenzeenee/t5-base-korean │
│ 번역           │ translation               │ Helsinki-NLP 시리즈       │
│ 이미지 분류     │ image-classification      │ google/vit-base-patch16  │
│ 객체 탐지       │ object-detection          │ facebook/detr-resnet-50  │
│ 질의 응답       │ question-answering        │ 원하는 QA 모델            │
└────────────────┴───────────────────────────┴──────────────────────────┘
```



### 2.3 모델 동작 확인

모델을 선택했으면, **서버 코드를 작성하기 전에** 노트북에서 먼저 동작을 확인합니다.

In [ ]:
# 예시: 텍스트 감정 분석
from transformers import pipeline

# 본인이 선택한 모델로 교체하세요
classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")

result = classifier("오늘 주가가 크게 올랐습니다")
print(result)
# [{'label': 'positive', 'score': 0.95}]

In [ ]:
# 예시: 이미지 분류
from transformers import pipeline

classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

result = classifier("test_image.jpg")
print(result)
# [{'label': 'tabby cat', 'score': 0.82}, ...]

> **이 셀이 정상 동작하면**, 서버 코드 작성으로 넘어갑니다.
> 여기서 에러가 나면, 모델을 바꾸거나 입력 형식을 확인하세요.



---

## 3. 프로젝트 뼈대 코드

---



### 3.1 폴더 구조

In [ ]:
import os

dirs = ["app", "models", "frontend"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("프로젝트 구조:")
print("""
my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt
""")

### 3.2 auth.py — 재사용

In [ ]:
%%writefile app/auth.py
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

### 3.3 schemas.py — 직접 작성

In [ ]:
%%writefile app/schemas.py
"""
입력/출력 스키마를 정의하세요.

참고: Day 2 섹션 4 (Pydantic 기초), Day 5 섹션 3 (HousingRequest)

TODO:
  - 본인의 모델 입력에 맞는 Request 스키마를 정의하세요.
  - 모델 출력에 맞는 Response 스키마를 정의하세요.
  - 필수 필드와 선택 필드를 구분하세요.
  - 적절한 검증 규칙(타입, 범위, 길이 등)을 추가하세요.

예시 (텍스트 분류의 경우):
  class PredictRequest(BaseModel):
      text: str = Field(..., min_length=1, max_length=5000)

  class PredictResponse(BaseModel):
      success: bool
      label: str
      confidence: float
"""
from pydantic import BaseModel, Field

# ── 여기에 작성하세요 ──────────────────────────

### 3.4 model_service.py — 직접 작성

In [ ]:
%%writefile app/model_service.py
"""
모델 로드와 추론 함수를 정의하세요.

참고: Day 1 섹션 5 (모델 로드), Day 5 섹션 2 (HousingPredictor)

TODO:
  1. load_model(): 모델을 로드하여 반환합니다.
     - transformers의 pipeline() 또는 from_pretrained()을 사용합니다.
     - 섹션 2.3에서 확인한 코드를 여기에 옮기면 됩니다.

  2. predict(model, input_data): 입력을 받아 추론 결과를 반환합니다.
     - 입력 전처리가 필요하면 여기서 합니다.
     - 결과를 dict로 반환합니다.

예시 (텍스트 분류의 경우):
  def load_model():
      return pipeline("text-classification", model="모델이름")

  def predict(model, text: str) -> dict:
      result = model(text)
      return {"label": result[0]["label"], "confidence": result[0]["score"]}
"""

# ── 여기에 작성하세요 ──────────────────────────

### 3.5 main.py — 직접 작성

In [ ]:
%%writefile app/main.py
"""
FastAPI 서버를 정의하세요.

참고: Day 5 섹션 3 (housing_api.py), Day 6 섹션 6 (image_api.py)

TODO:
  1. FastAPI 앱 생성
  2. startup 이벤트에서 모델 로드
  3. GET /health 엔드포인트
  4. POST /predict 엔드포인트
     - Pydantic 스키마로 입력 검증
     - Depends(verify_api_key)로 인증 적용
     - run_in_executor로 비동기 추론

필수 import:
  import asyncio
  from concurrent.futures import ThreadPoolExecutor
  from fastapi import FastAPI, Depends, HTTPException
  from app.auth import verify_api_key
  from app.schemas import PredictRequest, PredictResponse  (본인이 정의한 이름)
  from app.model_service import load_model, predict
"""

# ── 여기에 작성하세요 ──────────────────────────

### 3.6 frontend/app.py — 직접 작성

In [ ]:
%%writefile frontend/app.py
"""
Streamlit 프론트엔드를 정의하세요.

참고: Day 4 섹션 6 (대시보드), Day 5 섹션 4 (주택 가격 UI)

TODO:
  1. st.title()로 제목
  2. 사이드바에 API Key 입력
  3. 본인의 모델에 맞는 입력 위젯 (text_input, file_uploader 등)
  4. 버튼 클릭 시 requests.post()로 API 호출
  5. 결과 표시

실행 방법:
  streamlit run frontend/app.py
"""
import streamlit as st
import requests

# ── 여기에 작성하세요 ──────────────────────────

---

## 4. 작업 시간

---

### 4.1 권장 순서



```
Step 1. 모델 선택 + 노트북에서 동작 확인 (섹션 2.3)
        → "이 모델이 내 입력에 대해 결과를 반환하는가?"

Step 2. schemas.py 작성
        → "입력과 출력의 형태를 정의"

Step 3. model_service.py 작성
        → "모델 로드 + 추론 함수"

Step 4. main.py 작성
        → "FastAPI 서버 조립"

Step 5. 서버 실행 + Swagger UI 테스트
        → "API가 동작하는가?"

Step 6. frontend/app.py 작성
        → "Streamlit UI 연결"
```



### 4.2 서버 실행 (Step 5에서 사용)

In [ ]:
# ⚠️ 커널 재시작 후 실행하세요

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨 — http://localhost:8000/docs")

### 4.3 API 테스트 템플릿 (Step 5에서 사용)

In [ ]:
import requests

API_URL = "http://localhost:8000"
HEADERS = {"X-API-Key": "test-key-001"}

# health check
print(requests.get(f"{API_URL}/health").json())

# 추론 테스트 — 본인의 입력에 맞게 수정하세요
response = requests.post(
    f"{API_URL}/predict",
    json={"여기에": "본인의 입력"},   # ← 수정
    headers=HEADERS,
)
print(f"상태: {response.status_code}")
print(f"결과: {response.json()}")

### 4.4 막혔을 때 참고할 Day



```
"스키마를 어떻게 정의하지?"        → Day 2 섹션 4, Day 5 섹션 3
"FastAPI 서버 구조가 기억 안 나"   → Day 5 섹션 3, Day 6 섹션 6
"run_in_executor 사용법?"         → Day 3 섹션 4
"인증 적용 방법?"                  → Day 6 섹션 2
"Streamlit에서 API 호출?"         → Day 4 섹션 6, Day 5 섹션 4
"에러 처리?"                      → Day 3 섹션 5
```

---

## 5. 발표 및 회고

---

### 5.1 발표 (개인당 5분)

```
발표 항목:
  1. 어떤 도메인/태스크를 선택했는가?
  2. 어떤 모델을 사용했는가? (선택 이유)
  3. 데모 시연 (Swagger UI 또는 Streamlit)
  4. 구현하면서 가장 어려웠던 부분은?
```

### 5.2 회고

```
스스로 돌아보기:
  - Day 1~7 교안 없이 코드를 작성할 수 있었는가?
  - 어떤 부분에서 교안을 다시 찾아봤는가?
  - 다음에 다시 만든다면 무엇을 다르게 하겠는가?
```

---

## 6. 8일간의 여정 정리

---



### 6.1 Day 1의 문제 → Day 8의 해결

```
Day 1의 문제                           해결한 Day
──────────────────────────────        ──────────
라이브러리가 없음                       Day 1: requirements.txt
모델 구조 코드 필요                     Day 1: model_utils.py 모듈 분리
전처리 로직 누락                       Day 5/7: 전처리 파라미터 저장
비개발자가 사용할 수 없음               Day 4/5/7: Streamlit UI
누구나 API 호출 가능                   Day 6: API Key 인증
스스로 서비스를 만들 수 있는가?          Day 8: 자율 프로젝트 ✅
```



### 6.2 8일간 배운 기술 전체 지도

```
Day 1: 환경 세팅 + 모델 직렬화          "모델을 저장하고 불러온다"
Day 2: FastAPI + Pydantic              "모델을 API로 감싼다"
Day 3: 비동기 처리 + 에러/로깅          "안정적으로 돌아가게 한다"
Day 4: Streamlit + 시스템 아키텍처      "누구나 쓸 수 있게 한다"
Day 5: [프로젝트 1] 정형 데이터 서비스   "따라하며 조립한다"
Day 6: 인증 + 파일 업로드               "보안과 비정형 데이터를 다룬다"
Day 7: [프로젝트 2] 텍스트/이미지 서비스  "패턴을 반복하며 익힌다"
Day 8: [자율 프로젝트] 나만의 서비스      "스스로 만든다"
```



### 6.3 Next Step: MLOps로 가는 길

```
이 과정에서 배운 것:                 다음 과정에서 배울 것:
──────────────────                  ──────────────────
수동으로 서버 실행                    → Docker로 패키징
단일 서버에서 실행                    → 클라우드 배포 (AWS, GCP)
코드 변경 시 수동 재시작              → CI/CD 파이프라인 (자동 빌드/배포)
모델 버전 1개                        → 모델 버전 관리 (MLflow, DVC)
수동 모니터링 (로그 확인)             → 자동 모니터링 (Prometheus, Grafana)
```



> **"코드를 고칠 때마다 매번 서버를 재시작해야 하나요?"**
>
> 그 질문의 답이 MLOps입니다.
> CI/CD 파이프라인이 코드 변경을 감지하면 자동으로 빌드, 테스트, 배포합니다.
> 여러분은 코드를 커밋하기만 하면 됩니다.

---



### ✅ Day 8 최종 체크포인트

```
Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?
Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?
Q3. run_in_executor를 사용한 이유는 무엇입니까?
Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?
Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?
```

Q1. Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?
→ text 필드가 비어있거나(min_length=1 위반), 500자 초과(max_length=500 위반)인 경우
   422 Unprocessable Entity를 자동 반환. 타입이 str이 아닌 경우도 차단.

Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?
→ 인증 없이 누구나 /predict 엔드포인트를 호출 가능. 무제한 모델 추론으로
   서버 자원 고갈 및 무단 사용 위험.

Q3. run_in_executor를 사용한 이유는 무엇입니까?
→ PyTorch 추론(CPU-bound)은 동기 블로킹 작업. async 함수 내에서 직접 호출하면
   FastAPI 이벤트 루프 전체가 멈춤. ThreadPoolExecutor로 별도 스레드에 위임해
   다른 요청을 동시에 처리 가능하게 함.

Q4. Day 1~8 중 가장 많이 참고한 Day는?
→ Day 6 (인증 구조 auth.py 재사용) + Day 2 (Pydantic 스키마 설계)

Q5. 실제 배포에 추가로 필요한 것은?
→ Docker 패키징, 클라우드 서버(EC2 등), HTTPS, CI/CD 파이프라인, 모니터링(로그/알림)

---

### 📌 Day 8 요약 & 전체 과정 완료

```
오늘 한 일:
  ✅ Day 1~7의 기술을 조합하여 나만의 서비스를 직접 만들었습니다.
  ✅ 교안 없이 설계 → 구현 → 테스트를 경험했습니다.
  ✅ 8일간의 여정을 회고하고, MLOps로 가는 길을 확인했습니다.

8일간의 전체 성과:
  🎉 PyTorch / HuggingFace 모델을 API로 서빙할 수 있습니다.
  🎉 비개발자도 사용 가능한 웹 UI를 붙일 수 있습니다.
  🎉 인증, 에러 처리, 로깅으로 안정적인 서비스를 만들 수 있습니다.
  🎉 스스로 설계하고 구현할 수 있다는 자신감을 얻었습니다.
```

### 배점 기준  
1. DAY 8 최종 체크포인트  
2. 본인의 프로젝트를 소개하는 MD (구현 과정 묘사, 결과 스크린샷 등이 있어야 함)  
3. 회고  